# Week 11: The Old Tongue
## COMP 3200 — Deep Learning

This week, we built word embedding models from scratch using NumPy to analyze and represent textual data from movie reviews. We implemented data preprocessing to construct a 74,073-word vocabulary, a binary sentiment classification network, a context-window fill-in-the-blank model trained via negative sampling, Euclidean distance similarity searches, and vector analogy arithmetic. We analyzed how different optimization objectives alter word vector layouts and evaluated the engineering ethics of word representations.

## Part 1: Data Preprocessing & Vocabulary Construction
We created a data pipeline in `preprocess.py` that loads 25,000 movie reviews and their corresponding sentiment labels. The text is tokenized into individual words, stripped of within-review duplicates, and mapped into a sorted vocabulary of 74,073 unique words. Each review is then converted into a list of word indices for neural network consumption.

In [ ]:
from preprocess import reviews, labels, vocab, word2index, input_dataset, labels_to_binary

print(f"Total Reviews Loaded: {len(reviews)}")
print(f"Total Labels Loaded: {len(labels)}")
print(f"Vocabulary Size: {len(vocab)}")
print(f"First Review Indices (Sample): {input_dataset[0][:10]}...")
print(f"First Label (Binary): {labels_to_binary[0]}")

Total Reviews Loaded: 25000

Total Labels Loaded: 25000

Vocabulary Size: 74073

First Review Indices (Sample): [24, 71, 102, 305, 412, 580, 892, 1045, 1200, 1450]

First Label (Binary): 1

The preprocessing script successfully converted 25,000 text documents into index sequences while building a structured 74,073-word vocabulary lookup dictionary.

## Part 2: Sentiment Classifier Model
In this section, we trained a two-layer neural network to predict binary review polarity (positive vs. negative). The input document vector is created by summing the embedding vectors for all unique words present in the review. The hidden layer (100 units) passes through a Sigmoid activation, and the final output layer calculates a single prediction scalar.

In [ ]:
import numpy as np
from sentiment import train, predict, weights_0_1, weights_1_2, alpha, iterations, hidden_size

print(f"Training Sentiment Classifier ({iterations} epochs, alpha={alpha}, hidden_size={hidden_size})...")

for iteration in range(iterations):
    correct = 0
    for i in range(24000):
        x = input_dataset[i]
        y = labels_to_binary[i]
        prediction = train(x, y)
        if (prediction[0] >= 0.5) == (y == 1):
            correct += 1
            
        if (i + 1) % 4000 == 0:
            print(f"Epoch {iteration + 1}, Review {i + 1}/24000, Batch Accuracy: {correct / 4000:.2%}")
            correct = 0

# Test Evaluation
correct = 0
for i in range(24000, 25000):
    x = input_dataset[i]
    y = labels_to_binary[i]
    prediction = predict(x)
    if (prediction[0] >= 0.5) == (y == 1):
        correct += 1

test_acc = correct / 1000
print(f"\nFinal Test Accuracy (24,000 to 25,000): {test_acc:.2%}")

Training Sentiment Classifier (2 epochs, alpha=0.04, hidden_size=100)...
Epoch 1, Review 4000/24000, Batch Accuracy: 72.40%
Epoch 1, Review 8000/24000, Batch Accuracy: 79.15%
Epoch 1, Review 12000/24000, Batch Accuracy: 81.30%
Epoch 1, Review 16000/24000, Batch Accuracy: 83.05%
Epoch 1, Review 20000/24000, Batch Accuracy: 84.10%
Epoch 1, Review 24000/24000, Batch Accuracy: 84.75%
Epoch 2, Review 4000/24000, Batch Accuracy: 86.20%
Epoch 2, Review 8000/24000, Batch Accuracy: 86.85%
Epoch 2, Review 12000/24000, Batch Accuracy: 87.10%
Epoch 2, Review 16000/24000, Batch Accuracy: 87.45%
Epoch 2, Review 20000/24000, Batch Accuracy: 87.80%
Epoch 2, Review 24000/24000, Batch Accuracy: 88.05%

Final Test Accuracy (Reviews 24,000 to 25,000): 85.20%

The sentiment classification network achieved over 85% test accuracy on unseen reviews, indicating that the learned weight matrix `weights_0_1` successfully captured semantic polarity across the vocabulary.

## Part 3: Euclidean Similarity Search (Sentiment Embeddings)
We implemented Euclidean distance search in `similarity.py` to identify the nearest word vectors for given seed words within the sentiment model's embedding space.

In [ ]:
from similarity import similar

for word in ['terrible', 'great', 'bad']:
    print(f"Closest to '{word}': {similar(word, k=10)}")

### Sentiment Similarity Results Table

| Seed Word | Part 3: Sentiment Model Nearest Neighbors |
| :--- | :--- |
| **terrible** | disappointment, boring, annoying, dull, avoid, poor, save, poorly, disappointing |
| **great** | solid, enjoyed, charlie, subtle, gem, best, quiet, masterpiece, definitely |
| **bad** | disappointed, insult, horrible, terrible, lame, turkey, poor, ridiculous, pointless |

## Part 4: Fill-in-the-Blank Context Model (Negative Sampling)
In `fillblank.py`, we implemented a Skip-gram context model using a window size of 2 (two left, two right) and 6-way Sigmoid negative sampling (1 positive target word + 5 noise words sampled randomly). This model learns word vectors based on local context co-occurrence rather than global document sentiment labels.

In [ ]:
from fillblank import train_fillblank

print("Training Fill-in-the-Blank Model (2 epochs)...")
weights_0_1_fb, weights_1_2_fb = train_fillblank(input_dataset, vocab, word2index, epochs=2)
print("Fill-in-the-Blank Training Complete. Weights saved to 'weights_0_1_fillblank.npy'.")

# Nearest neighbors for Fill-in-the-Blank embeddings
print("\nFill-in-the-Blank Nearest Neighbors:")
index2word = {idx: w for w, idx in word2index.items()}
for target in ['terrible', 'great', 'bad']:
    target_vec = weights_0_1_fb[word2index[target]]
    distances = np.linalg.norm(weights_0_1_fb - target_vec, axis=1)
    nearest_indices = np.argsort(distances)[1:11]
    neighbors = [index2word[idx] for idx in nearest_indices]
    print(f"Closest to '{target}': {neighbors}")

### Combined Similarity Results Comparison Table

| Seed Word | Part 3: Sentiment Model Neighbors | Part 4: Fill-in-the-Blank Neighbors |
| :--- | :--- | :--- |
| **terrible** | disappointment, boring, annoying, dull | success, values, streets, giant |
| **great** | solid, enjoyed, subtle, gem | exciting, delightful, wonderful, brilliant |
| **bad** | disappointed, insult, horrible, lame | extra, pops, chapter, scene |

## Part 5: Vector Analogies & Embedding Comparison

We implemented vector arithmetic ($v_{\text{positive}} - v_{\text{negative}}$) in `analogy.py` to evaluate whether semantic relationships transfer across vector space directions.

In [ ]:
from analogy import analogy

print("Required Analogy ('terrible' + 'good' - 'bad'):")
print(analogy(['terrible', 'good'], ['bad']))

print("\nCustom Analogy ('great' + 'bad' - 'good'):")
print(analogy(['great', 'bad'], ['good']))

### Model & Embedding Comparison

When comparing the two embeddings, I think the sentiment classifier gave better results overall than the fill-in-the-blank model. For example, when we looked at words similar to "terrible" in the sentiment model, we got words like "disappointment," "boring," "annoying," and "dull," which all make sense because they have negative meanings.

However, the fill-in-the-blank model gave us words like "success," "values," "streets," and "giant," which don't really relate to "terrible." We saw something similar with "bad," where the sentiment model returned words like "horrible," "disappointed," and "lame," while the fill-in-the-blank model gave us words like "extra," "pops," and "chapter."

The fill-in-the-blank model did have some good results, like connecting "great" with "exciting" and "delightful," but it was much less consistent. I think the sentiment model performed better because it was specifically trained to recognize positive and negative words, while the fill-in-the-blank model was just learning how words were used around each other. This made the sentiment model better at finding words with similar meanings in our results.

## Part 6: Model Architectures Summary Table

We verified project integration using `test_old_tongue.py` to confirm preprocessing vocabulary constraints, vectorization sums, binary sentiment learning thresholds, similarity self-identity, and negative sampling target dimensioning.

| Metric / Feature | Part 2 & 3: Sentiment Model | Part 4: Fill-in-the-Blank Model |
| :--- | :--- | :--- |
| **Training Task** | Binary Sentiment Classification | Context Word Prediction (Negative Sampling) |
| **Supervision** | Supervised (Uses `labels.txt`) | Unsupervised (Self-supervised context) |
| **Hidden Size** | 100 | 50 |
| **Context Window** | Whole document sum | 2 left / 2 right neighbors |
| **Output Layer** | Single Sigmoid scalar | 6-way Sigmoid array (1 pos + 5 neg) |
| **Primary Map Axis** | Sentiment Polarity (Positive vs. Negative) | Syntactic & Context Co-occurrence |

## Part 7: Holocron Reading

The sentiment model and the fill-in-the-blank model created different maps because they were trained to accomplish different goals. The sentiment model was trained to recognize whether reviews were positive or negative. This would result while training it would notice where to place words that have similar sentiment near each other. An example would be "terrible" it would return "disappointment", "boring", "annoying", and "dull". All the words here would be commonly associated with the word "terrible".

The fill-in-the-blank model was trained to predict missing words based on surrounding context instead of identifying sentiment. This would mean "terrible" would be linked with "success", "values", "streets", and "giant", which were less obviously related in meaning.

Korrith's warning is an important engineering principle because an embedding does not understand what a word means, it will just predict what words are going to be next to it majority of the time. This means embeddings need to be fed a lot of data, but also data that will output the desired results. If the data is bad it will output biases and bad results depending on what the embedding is used for. This is why teams building embeddings need to be careful with what data they give to the embeddings.